# Regression Analysis

Models for binary outcomes (`qual_changed`, `suspense`): **logit** with average marginal effects.  
Model for count outcome (`qual_count`): **Poisson**.  
Treatment: `fifa_rule` = 1 for World Cup, 0 for European Championship.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit, Poisson
from getpass import getuser

user = getuser()
base = rf'C:\Users\{user}\Documents\GitHub\tb_football'

goals = pd.read_excel(rf'{base}\data\out\goals_merged.xlsx')
mbm   = pd.read_excel(rf'{base}\data\out\mbm_merged.xlsx')

goals['group_id'] = goals['year'].astype(str) + '_' + goals['stage'].astype(str) + '_' + goals['fifa_rule'].astype(str)
mbm['group_id']   = mbm['year'].astype(str) + '_' + mbm['stage'].astype(str) + '_' + mbm['fifa_rule'].astype(str)

goals['year_c'] = goals['year'] - 2000
mbm['year_c']   = mbm['year'] - 2000

print(f'goals: {len(goals)} rows  |  mbm: {len(mbm)} rows')
print(f'WC groups: {goals[goals.fifa_rule==1].group_id.nunique()}  |  EU groups: {goals[goals.fifa_rule==0].group_id.nunique()}')

In [ ]:
def logit_cluster(df, y_col, x_cols, cluster_col):
    """Logit with cluster-robust SE. Returns (model, margins) or (None, None) if insufficient data."""
    sub = df[[y_col, cluster_col] + x_cols].dropna()
    if len(sub) == 0 or sub[y_col].nunique() < 2:
        return None, None
    y = sub[y_col]
    X = sm.add_constant(sub[x_cols])
    groups = sub[cluster_col]
    model = Logit(y, X).fit(cov_type='cluster', cov_kwds={'groups': groups}, disp=0)
    margins = model.get_margeff()
    return model, margins


def show_margins(margins, model, title=''):
    if title:
        print(f'\n{"="*60}')
        print(title)
        print('='*60)
    if model is None:
        print('  [skipped — insufficient observations]')
        return
    tbl = pd.DataFrame({
        'dy/dx': margins.margeff.round(4),
        'se':    margins.margeff_se.round(4),
        'z':     margins.tvalues.round(2),
        'p':     margins.pvalues.round(3),
    }, index=margins.summary_frame().index)
    print(tbl.to_string())
    print(f'N={int(model.nobs)}  Pseudo-R²={model.prsquared:.3f}')


def poisson_reg(df, y_col, x_cols):
    sub = df[[y_col] + x_cols].dropna()
    if len(sub) == 0:
        return None
    y = sub[y_col]
    X = sm.add_constant(sub[x_cols])
    model = Poisson(y, X).fit(disp=0)
    return model


def show_poisson(model, title=''):
    if title:
        print(f'\n{"="*60}')
        print(title)
        print('='*60)
    if model is None:
        print('  [skipped — insufficient observations]')
        return
    tbl = pd.DataFrame({
        'coef': model.params.round(4),
        'se':   model.bse.round(4),
        'z':    model.tvalues.round(2),
        'p':    model.pvalues.round(3),
    })
    print(tbl.to_string())
    print(f'N={int(model.nobs)}  LogL={model.llf:.1f}')

---
## Goals dataset

### Model 1 — Logit, average marginal effects
`qual_changed = f(fifa_rule, elo_favorite, elo_underdog)`  
Clustered SE at group (year × stage) level.

In [ ]:
m1_vars = ['fifa_rule', 'elo_favorite', 'elo_underdog']

m1_main, m1_me = logit_cluster(goals, 'qual_changed', m1_vars, 'group_id')
show_margins(m1_me, m1_main, 'Model 1 — Main (avg marginal effects)')

### Model 2 — Logit, average marginal effects
`qual_changed = f(fifa_rule, elo1st, elo2nd, elo3rd, elo4th)`  
Clustered SE at group level.

In [ ]:
m2_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']

m2_main, m2_me = logit_cluster(goals, 'qual_changed', m2_vars, 'group_id')
show_margins(m2_me, m2_main, 'Model 2 — Main (avg marginal effects)')

---
## Robustness checks (Models 1 & 2)

Each check adds `year` and `year × fifa_rule` to the baseline regressors.

> **Why not year fixed effects?**  
> The World Cup and European Championship happen in alternating years — each year hosts exactly one tournament type. A full set of year dummies would therefore be collinear with `fifa_rule`, making the treatment coefficient unidentified. Adding `year` as a continuous variable captures a smooth time trend while leaving the between-tournament contrast to `fifa_rule`. The interaction `year × fifa_rule` allows the trend to differ across tournaments without absorbing the treatment effect.

In [ ]:
# Elo common-support bounds
# Keep groups whose elo range falls within the elo support shared by both WC and EU.
# Lower bound: highest elo of the weakest team across both tournaments (common floor).
# Upper bound: lowest elo of the strongest team across both tournaments (common ceiling).
wc_elo_min = goals[goals.fifa_rule==1][['elo1st','elo2nd','elo3rd','elo4th']].min().min()
wc_elo_max = goals[goals.fifa_rule==1][['elo1st','elo2nd','elo3rd','elo4th']].max().max()
eu_elo_min = goals[goals.fifa_rule==0][['elo1st','elo2nd','elo3rd','elo4th']].min().min()
eu_elo_max = goals[goals.fifa_rule==0][['elo1st','elo2nd','elo3rd','elo4th']].max().max()

elo_lower = max(wc_elo_min, eu_elo_min)
elo_upper = min(wc_elo_max, eu_elo_max)
print(f'WC elo range: [{wc_elo_min:.0f}, {wc_elo_max:.0f}]')
print(f'EU elo range: [{eu_elo_min:.0f}, {eu_elo_max:.0f}]')
print(f'Common support: [{elo_lower:.0f}, {elo_upper:.0f}]')

goals['g_elo_min'] = goals.groupby('group_id')['elo4th'].transform('first')
goals['g_elo_max'] = goals.groupby('group_id')['elo1st'].transform('first')
elo_mask = (goals['g_elo_min'] >= elo_lower) & (goals['g_elo_max'] <= elo_upper)
print(f'Groups in common support: {goals[elo_mask].group_id.nunique()} of {goals.group_id.nunique()}')

filters = {
    'Two qualifying teams':   goals['third_qualify'] == 0,
    'Three qualifying teams': goals['third_qualify'] == 1,
    '2-points rule (<=1992)': goals['year'] <= 1992,
    '3-points rule (>1992)':  goals['year'] > 1992,
    'Elo common support':     elo_mask,
}

In [ ]:
# Model 1 robustness
m1_robust_vars = ['fifa_rule', 'elo_favorite', 'elo_underdog', 'year_c']

m, me = logit_cluster(goals, 'qual_changed', m1_robust_vars, 'group_id')
show_margins(me, m, 'Model 1 + year_c')

for label, mask in filters.items():
    m, me = logit_cluster(goals[mask], 'qual_changed', m1_robust_vars, 'group_id')
    show_margins(me, m, f'Model 1 Robustness: {label}')

In [ ]:
# Model 2 robustness
m2_robust_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th', 'year_c']

m, me = logit_cluster(goals, 'qual_changed', m2_robust_vars, 'group_id')
show_margins(me, m, 'Model 2 + year_c')

for label, mask in filters.items():
    m, me = logit_cluster(goals[mask], 'qual_changed', m2_robust_vars, 'group_id')
    show_margins(me, m, f'Model 2 Robustness: {label}')

---
## Model 3 — Poisson count (group level)

Collapse to one row per group using `max(qual_count)`.  
`qual_count = f(fifa_rule, elo1st, elo2nd, elo3rd, elo4th)` — coefficients are log-rate ratios.

In [ ]:
goals_group = (
    goals.groupby(['year', 'stage', 'fifa_rule'])
    .agg(
        qual_count=('qual_count', 'max'),
        elo1st=('elo1st', 'first'), elo2nd=('elo2nd', 'first'),
        elo3rd=('elo3rd', 'first'), elo4th=('elo4th', 'first'),
        year_c=('year_c', 'first'),
        third_qualify=('third_qualify', 'max'),
        g_elo_min=('g_elo_min', 'first'), g_elo_max=('g_elo_max', 'first'),
    )
    .reset_index()
)
print(f'Group-level dataset: {len(goals_group)} groups')
print(goals_group.groupby('fifa_rule')['qual_count'].describe().round(2))

In [ ]:
m3_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']
show_poisson(poisson_reg(goals_group, 'qual_count', m3_vars), 'Model 3 Poisson — Main')

In [ ]:
m3_robust_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th', 'year_c']

show_poisson(poisson_reg(goals_group, 'qual_count', m3_robust_vars), 'Model 3 + year_c')

group_filters = {
    'Two qualifying teams':   goals_group['third_qualify'] == 0,
    'Three qualifying teams': goals_group['third_qualify'] == 1,
    '2-points rule (<=1992)': goals_group['year'] <= 1992,
    '3-points rule (>1992)':  goals_group['year'] > 1992,
    'Elo common support':     (goals_group['g_elo_min'] >= elo_lower) & (goals_group['g_elo_max'] <= elo_upper),
}

for label, mask in group_filters.items():
    show_poisson(poisson_reg(goals_group[mask], 'qual_count', m3_robust_vars), f'Model 3 Robustness: {label}')

---
## Models 4 & 5 — Logit for suspense (goal level)

Using the **goals dataset**, `suspense` is the binary indicator that a single additional goal
would change the set of qualifying teams.

- **Model 4**: `suspense = f(fifa_rule, elo_fav, elo_und)`
- **Model 5**: `suspense = f(fifa_rule, elo1st, elo2nd, elo3rd, elo4th)`

Clustered SE at group (year × stage) level.

In [ ]:
# Model 4: suspense ~ fifa_rule + elo_fav + elo_und (goals dataset)
m4_vars = ['fifa_rule', 'elo_favorite', 'elo_underdog']

m4_main, m4_me = logit_cluster(goals, 'suspense', m4_vars, 'group_id')
show_margins(m4_me, m4_main, 'Model 4 — Main (avg marginal effects)')

In [ ]:
# Model 4 robustness
m4_robust_vars = ['fifa_rule', 'elo_favorite', 'elo_underdog', 'year_c']

m, me = logit_cluster(goals, 'suspense', m4_robust_vars, 'group_id')
show_margins(me, m, 'Model 4 + year_c')

for label, mask in filters.items():
    m, me = logit_cluster(goals[mask], 'suspense', m4_robust_vars, 'group_id')
    show_margins(me, m, f'Model 4 Robustness: {label}')

### Model 5 — Logit for suspense, group-level Elo
`suspense = f(fifa_rule, elo1st, elo2nd, elo3rd, elo4th)`

In [ ]:
# Model 5: suspense ~ fifa_rule + elo1st-4th (goals dataset)
m5_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']

m5_main, m5_me = logit_cluster(goals, 'suspense', m5_vars, 'group_id')
show_margins(m5_me, m5_main, 'Model 5 — Main (avg marginal effects)')

In [ ]:
# Model 5 robustness
m5_robust_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th', 'year_c']

m, me = logit_cluster(goals, 'suspense', m5_robust_vars, 'group_id')
show_margins(me, m, 'Model 5 + year_c')

for label, mask in filters.items():
    m, me = logit_cluster(goals[mask], 'suspense', m5_robust_vars, 'group_id')
    show_margins(me, m, f'Model 5 Robustness: {label}')

---
## MBM dataset — Suspense model

`suspense = f(fifa_rule, elo1st, elo2nd, elo3rd, elo4th)` — logit, average marginal effects.  
Clustered SE at group level.

In [ ]:
# Elo common-support bounds for mbm (same logic as goals)
wc_elo_min_m = mbm[mbm.fifa_rule==1][['elo1st','elo2nd','elo3rd','elo4th']].min().min()
wc_elo_max_m = mbm[mbm.fifa_rule==1][['elo1st','elo2nd','elo3rd','elo4th']].max().max()
eu_elo_min_m = mbm[mbm.fifa_rule==0][['elo1st','elo2nd','elo3rd','elo4th']].min().min()
eu_elo_max_m = mbm[mbm.fifa_rule==0][['elo1st','elo2nd','elo3rd','elo4th']].max().max()

elo_lower_m = max(wc_elo_min_m, eu_elo_min_m)
elo_upper_m = min(wc_elo_max_m, eu_elo_max_m)
print(f'MBM common support: [{elo_lower_m:.0f}, {elo_upper_m:.0f}]')

mbm['g_elo_min'] = mbm.groupby('group_id')['elo4th'].transform('first')
mbm['g_elo_max'] = mbm.groupby('group_id')['elo1st'].transform('first')
elo_mask_m = (mbm['g_elo_min'] >= elo_lower_m) & (mbm['g_elo_max'] <= elo_upper_m)
print(f'MBM groups in common support: {mbm[elo_mask_m].group_id.nunique()} of {mbm.group_id.nunique()}')

mbm_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']

m, me = logit_cluster(mbm, 'suspense', mbm_vars, 'group_id')
show_margins(me, m, 'MBM Model — Main (avg marginal effects)')

In [ ]:
mbm_robust_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th', 'year_c']

m, me = logit_cluster(mbm, 'suspense', mbm_robust_vars, 'group_id')
show_margins(me, m, 'MBM + year_c')

mbm_filters = {
    'Two qualifying teams':   mbm['third_qualify'] == 0,
    'Three qualifying teams': mbm['third_qualify'] == 1,
    '2-points rule (<=1992)': mbm['year'] <= 1992,
    '3-points rule (>1992)':  mbm['year'] > 1992,
    'Elo common support':     elo_mask_m,
}

for label, mask in mbm_filters.items():
    m, me = logit_cluster(mbm[mask], 'suspense', mbm_robust_vars, 'group_id')
    show_margins(me, m, f'MBM Robustness: {label}')

### MBM Model B — Logit for qualification change (MBM dataset)
`qual_changed = f(fifa_rule, elo1st, elo2nd, elo3rd, elo4th)`  
Clustered SE at group level.

In [ ]:
# MBM Model B: qual_changed on minute-by-minute dataset
mbm_b_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']

mbm_b_main, mbm_b_me = logit_cluster(mbm, 'qual_changed', mbm_b_vars, 'group_id')
show_margins(mbm_b_me, mbm_b_main, 'MBM Model B — Main (avg marginal effects)')

In [ ]:
# MBM Model B robustness
mbm_b_robust_vars = ['fifa_rule', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th', 'year_c']

m, me = logit_cluster(mbm, 'qual_changed', mbm_b_robust_vars, 'group_id')
show_margins(me, m, 'MBM Model B + year_c')

for label, mask in mbm_filters.items():
    m, me = logit_cluster(mbm[mask], 'qual_changed', mbm_b_robust_vars, 'group_id')
    show_margins(me, m, f'MBM Model B Robustness: {label}')